# Stage 56: Pointwise second moments for residual even integers

This reproducible SageMath notebook studies the finite residual set
$H=\{N\ge 4: N\text{ even},\ N-1\text{ composite},\ N/2\text{ composite}\}$.
It counts unordered distinct-prime representations, computes the exact
modulo-210 admissibility factor, evaluates rational moments, and checks the
four-prime expansion by two independent finite enumerations.

All arithmetic and primality decisions in the experiments are exact. The
results concern only the displayed finite ranges. They do not prove Goldbach's
conjecture or coverage of every integer in $H$.


In [16]:
from sage.all import QQ, ZZ, is_prime
from sage.version import version as SAGE_VERSION
from math import gcd

WHEEL_PRIMES = (2, 3, 5, 7)
WHEEL = 210
PHI_WHEEL = 48

print('SageMath version:', SAGE_VERSION)
print('Wheel:', WHEEL, '=', '*'.join(map(str, WHEEL_PRIMES)))


SageMath version: 10.9
Wheel: 210 = 2*3*5*7


## Definitions and exact wheel factor

For even $N$, let $A_{210}(N)$ be the number of residue classes $a\pmod{210}$
for which both $a$ and $N-a$ are coprime to 210. By the Chinese remainder
theorem,

$$A_{210}(N)=\prod_{\ell\in\{2,3,5,7\}}(\ell-\nu_\ell(N)),\qquad
\nu_\ell(N)=\begin{cases}1,&\ell\mid N,\\2,&\ell\nmid N.\end{cases}$$

The truncated singular factor is
$\mathfrak S_{210}(N)=(A_{210}(N)/210)/(48/210)^2=210A_{210}(N)/48^2$.
The normalized count used below is $X_N=R(N)/A_{210}(N)$, where $R(N)$ counts
unordered pairs $p<q$ of distinct primes with $p+q=N$. This is a finite-wheel
normalization, not an asymptotic formula for $R(N)$.


In [17]:
def is_residual(N):
    N = int(N)
    return (N >= 4 and N % 2 == 0
            and not is_prime(N - 1)
            and not is_prime(N // 2))


def prime_pairs(N):
    """Return each genuine unordered distinct-prime pair exactly once."""
    N = int(N)
    return [(p, N - p)
            for p in range(2, N // 2 + 1)
            if p < N - p and is_prime(p) and is_prime(N - p)]


def admissibility_data(N):
    N = int(N)
    if N % 2:
        raise ValueError('The wheel factor here is defined for even N.')
    return [(ell, 1 if N % ell == 0 else 2) for ell in WHEEL_PRIMES]


def admissible_wheel_count(N):
    a = 1
    for ell, nu in admissibility_data(N):
        a *= ell - nu
    return ZZ(a)


def direct_wheel_count(N):
    N = int(N)
    return ZZ(sum(1 for a in range(WHEEL)
                  if gcd(a, WHEEL) == 1 and gcd(N - a, WHEEL) == 1))


def truncated_singular_factor(N):
    return QQ(WHEEL * admissible_wheel_count(N)) / (PHI_WHEEL ** 2)


def normalized_count(N):
    return QQ(len(prime_pairs(N))) / admissible_wheel_count(N)

wheel_examples = [4, 6, 10, 30, 42, 50, 100, 200]
print('N | local (ell, nu_ell) data | A_210(N) | singular factor')
for n in wheel_examples:
    a = admissible_wheel_count(n)
    assert a == direct_wheel_count(n)
    print(f'{n} | {admissibility_data(n)} | {a} | {truncated_singular_factor(n)}')
print('Product formula agrees with direct residue enumeration for every row.')


N | local (ell, nu_ell) data | A_210(N) | singular factor
4 | [(2, 1), (3, 2), (5, 2), (7, 2)] | 15 | 175/128
6 | [(2, 1), (3, 1), (5, 2), (7, 2)] | 30 | 175/64
10 | [(2, 1), (3, 2), (5, 1), (7, 2)] | 20 | 175/96
30 | [(2, 1), (3, 1), (5, 1), (7, 2)] | 40 | 175/48
42 | [(2, 1), (3, 1), (5, 2), (7, 1)] | 36 | 105/32
50 | [(2, 1), (3, 2), (5, 1), (7, 2)] | 20 | 175/96
100 | [(2, 1), (3, 2), (5, 1), (7, 2)] | 20 | 175/96
200 | [(2, 1), (3, 2), (5, 1), (7, 2)] | 20 | 175/96
Product formula agrees with direct residue enumeration for every row.


## Exact block moments and the variance criterion

For a finite block $B\subset H$ of size $k$, compute
$\mu_B=k^{-1}\sum_{N\in B}X_N$ and
$M_{2,B}=k^{-1}\sum_{N\in B}X_N^2$ exactly in $\mathbb Q$. The centered
sum is $V_B=\sum(X_N-\mu_B)^2=k(M_{2,B}-\mu_B^2)$. The proposed sufficient
coverage test is

$$V_B<\frac{k}{k-1}\mu_B^2.$$

Indeed, if one $X_N=0$, Cauchy--Schwarz applied to the other $k-1$ values gives
the reverse weak inequality. Thus passing the strict test certifies positive
counts throughout that finite block. Failure of the test does not imply a zero.


In [18]:
# Cache all pointwise data through the largest search endpoint.
MAX_N = 2000
point_data = {}
for n in range(4, MAX_N + 1, 2):
    if is_residual(n):
        pairs = prime_pairs(n)
        A = admissible_wheel_count(n)
        point_data[n] = {
            'R': ZZ(len(pairs)), 'A': A, 'X': QQ(len(pairs)) / A,
            'pairs': pairs,
        }
all_residuals = sorted(point_data)
print('Residual integers in [4, 2000]:', len(all_residuals))
print('Zero pair counts in [4, 2000]:',
      sum(point_data[n]['R'] == 0 for n in all_residuals))


def block_statistics(low, high):
    ns = [n for n in range(low + (low % 2), high + 1, 2)
          if n in point_data]
    k = len(ns)
    if k < 2:
        raise ValueError('Each reported block must contain at least two residual integers.')
    xs = [point_data[n]['X'] for n in ns]
    mean = sum(xs, QQ(0)) / k
    second = sum((x*x for x in xs), QQ(0)) / k
    variance_sum = sum(((x - mean)**2 for x in xs), QQ(0))
    threshold = (QQ(k) / (k - 1)) * mean**2
    assert variance_sum == k * (second - mean**2)
    return {
        'low': low, 'high': high, 'N': ns, 'k': k,
        'zeros': sum(point_data[n]['R'] == 0 for n in ns),
        'mean': mean, 'second': second, 'variance_sum': variance_sum,
        'threshold': threshold, 'passes': variance_sum < threshold,
        'ratio': variance_sum / threshold if threshold else None,
    }

blocks = [(50, 100), (50, 200), (202, 500), (502, 1000), (1002, 2000)]
block_results = [block_statistics(lo, hi) for lo, hi in blocks]
print('Block | k | zero counts | exact mean | exact second moment | exact V | exact threshold | criterion')
for b in block_results:
    print(f"{b['low']}-{b['high']} | {b['k']} | {b['zeros']} | {b['mean']} | "
          f"{b['second']} | {b['variance_sum']} | {b['threshold']} | {b['passes']}")


Residual integers in [4, 2000]: 564
Zero pair counts in [4, 2000]: 0
Block | k | zero counts | exact mean | exact second moment | exact V | exact threshold | criterion
50-100 | 12 | 0 | 353/1440 | 2173/34560 | 5771/172800 | 124609/1900800 | True
50-200 | 33 | 0 | 3991/11880 | 34559/285120 | 18416/66825 | 15928081/136857600 | False
202-500 | 78 | 0 | 7193/11232 | 5746411/13478400 | 51178949/40435200 | 51739249/124540416 | False
502-1000 | 143 | 0 | 927/880 | 84441553/74131200 | 11991823/2851200 | 11171277/9996800 | False
1002-2000 | 306 | 0 | 93707/55080 | 235130029/79315200 | 850887041/39657600 | 8781001849/3023892000 | False


## Active search for an overly strong variance bound

The same strict inequality is scanned over every even-starting 100-integer
window with start from 50 through 900. This search looks for finite blocks that
violate the proposed inequality, including blocks where direct exact counting
still finds no zero. Such a violation refutes the inequality as a universal
bound; it does not refute the one-way coverage implication proved above.


In [19]:
window_results = []
for lo in range(50, 901, 2):
    b = block_statistics(lo, lo + 100)
    window_results.append(b)

violations = [b for b in window_results if not b['passes']]
if violations:
    worst = max(violations, key=lambda b: b['ratio'])
    print('Windows scanned:', len(window_results))
    print('Windows violating V < k/(k-1) * mean^2:', len(violations))
    print('Largest exact ratio V / threshold:', worst['ratio'])
    print('Counterexample window:', (worst['low'], worst['high']))
    print('Residual count k:', worst['k'], 'zero representations:', worst['zeros'])
    print('Exact mean:', worst['mean'])
    print('Exact second moment:', worst['second'])
    print('Exact centered sum:', worst['variance_sum'])
    print('Exact proposed threshold:', worst['threshold'])
else:
    print('No violating window found in this finite search.')

# Verify the zero-implies-reverse-inequality lemma on every searched window.
for b in window_results:
    if b['zeros']:
        assert b['variance_sum'] >= b['threshold']
print('Finite checks of the zero-implies-reverse-inequality lemma passed.')


Windows scanned: 426
Windows violating V < k/(k-1) * mean^2: 3
Largest exact ratio V / threshold: 1716160/1568187
Counterexample window: (50, 150)
Residual count k: 21 zero representations: 0
Exact mean: 241/840
Exact second moment: 6751/77760
Exact centered sum: 5363/56700
Exact proposed threshold: 58081/672000
Finite checks of the zero-implies-reverse-inequality lemma passed.


## Diagonal and off-diagonal four-prime correlations

Let $\mathcal P_N=\{(p,q):p<q,\ p,q\text{ prime},\ p+q=N\}$. Then

$$R(N)^2=\#\{(p,q,r,s): (p,q),(r,s)\in\mathcal P_N\}
= D(N)+O(N),$$

where the ordered pair-of-representations diagonal has $D(N)=R(N)$ and the
off-diagonal has $O(N)=R(N)(R(N)-1)$. Distinct representations cannot share
a prime: a shared prime forces the other summand and hence the same unordered
pair. Thus every off-diagonal term is a four-distinct-prime correlation.
For the normalized second moment, each term is weighted by $A_{210}(N)^{-2}$.


In [20]:
def four_prime_by_pair_cross_product(N):
    pairs = prime_pairs(N)
    diagonal = sum(1 for left in pairs for right in pairs if left == right)
    off_diagonal = sum(1 for left in pairs for right in pairs if left != right)
    return ZZ(diagonal + off_diagonal), ZZ(diagonal), ZZ(off_diagonal)


def four_prime_by_direct_quadruple_scan(N):
    # Independent enumeration: scan the two lower primes directly and test all
    # four entries, without calling prime_pairs or using its returned list.
    total = diagonal = off_diagonal = 0
    for p in range(2, N // 2 + 1):
        q = N - p
        if p >= q or not is_prime(p) or not is_prime(q):
            continue
        for r in range(2, N // 2 + 1):
            s = N - r
            if r >= s or not is_prime(r) or not is_prime(s):
                continue
            total += 1
            if p == r and q == s:
                diagonal += 1
            else:
                off_diagonal += 1
    return ZZ(total), ZZ(diagonal), ZZ(off_diagonal)

verify_ns = [n for n in point_data if 50 <= n <= 200]
total_diagonal = total_off_diagonal = total_four_prime = 0
for n in verify_ns:
    first = four_prime_by_pair_cross_product(n)
    second = four_prime_by_direct_quadruple_scan(n)
    R = point_data[n]['R']
    assert first == second
    assert first == (R**2, R, R*(R-1))
    total_four_prime += first[0]
    total_diagonal += first[1]
    total_off_diagonal += first[2]

print('Residual N independently checked:', len(verify_ns))
print('Aggregate four-prime count:', total_four_prime)
print('Aggregate diagonal count:', total_diagonal)
print('Aggregate off-diagonal count:', total_off_diagonal)
assert total_four_prime == total_diagonal + total_off_diagonal
print('Both exact algorithms agree pointwise and in the aggregate.')


Residual N independently checked: 33
Aggregate four-prime count: 1803
Aggregate diagonal count: 229
Aggregate off-diagonal count: 1574
Both exact algorithms agree pointwise and in the aggregate.


In [21]:
# Exact decomposition of the normalized second moment over H intersect [50, 200].
verify_block = verify_ns
K_verify = len(verify_block)
diagonal_weight = sum((QQ(point_data[n]['R']) / point_data[n]['A']**2
                       for n in verify_block), QQ(0))
off_diagonal_weight = sum((QQ(point_data[n]['R']*(point_data[n]['R']-1))
                           / point_data[n]['A']**2
                           for n in verify_block), QQ(0))
second_moment_sum = sum((point_data[n]['X']**2 for n in verify_block), QQ(0))
assert second_moment_sum == diagonal_weight + off_diagonal_weight
print('H in [50, 200]: k =', K_verify)
print('Unweighted diagonal sum ΣR(N):', total_diagonal)
print('Unweighted off-diagonal sum ΣR(N)(R(N)-1):', total_off_diagonal)
print('Exact weighted diagonal contribution:', diagonal_weight)
print('Exact weighted off-diagonal contribution:', off_diagonal_weight)
print('Exact Σ X_N^2:', second_moment_sum)
print('Identity verified:', second_moment_sum == diagonal_weight + off_diagonal_weight)


H in [50, 200]: k = 33
Unweighted diagonal sum ΣR(N): 229
Unweighted off-diagonal sum ΣR(N)(R(N)-1): 1574
Exact weighted diagonal contribution: 76033/129600
Exact weighted off-diagonal contribution: 27647/8100
Exact Σ X_N^2: 34559/8640
Identity verified: True


## Interpretation and limitations

The definitions, wheel product, moment identities, and four-prime expansion are
exact mathematical identities. The printed counts and variance outcomes are
finite computational observations for the stated ranges, evaluated with Sage's
exact primality test and rational arithmetic.

The modulo-210 factor records only local admissibility at 2, 3, 5, and 7. It is
not the full singular series and gives no error bound for replacing prime-pair
counts by a main term. The second-moment criterion is a sufficient finite-block
coverage condition; its failure is inconclusive. Establishing coverage for all
$N\in H$ would require a proof that the pointwise count is positive for every
such $N$, or a valid global argument that forces the criterion on every block.
The finite search here supplies neither. No claim is made to have proved
Goldbach's conjecture or universal coverage.
